# Distinguishing Signal- and Prompt-like $3\pi$ Production for a Measurement of $R(D^*)$

This notebook introduces methods for distinguishing $B^0 \to D^{*-}\tau^+\nu_\tau$, with $\tau^+\to\pi^+\pi^-\pi^+\bar\nu_\tau$, from events in which the three charged pions are produced promptly at the $B$ decay vertex. The methods explored include two machine learning architectures--boosted decision trees (BDT), in particular an implementation called XGBoost, and neural networks--and are evaluated relative to a more typical cut-based approach. These architectures differ in complexity and have various pros/cons that will be explored through the activity.

## Physics motivation

Lepton flavour universality (LFU)  is the Standard Model statement that the electroweak interaction couples in the same way to the three charged leptons $e, \mu, \tau$, apart from effects caused by their different masses. One important quantity to measure in order to test LFU is

$$
R(D^*) = \frac{\mathcal{B}(B^0\to D^{*-}\tau^+\nu_\tau)}{\mathcal{B}(B^0\to D^{*-}\mu^+\nu_\mu)}.
$$

This activity is based on [a 2024 measurement](https://arxiv.org/pdf/2305.01463) of $R(D^*)$ using the $\tau^+\to\pi^+\pi^-\pi^+\bar\nu_\tau$ decay with LHCb Run 2 data. The signal decay chain is

$$
B^0\to D^{*-}\tau^+\nu_\tau,\qquad
D^{*-}\to \bar D^0\pi^-,\qquad
\bar D^0\to K^+\pi^-,\qquad
\tau^+\to\pi^+\pi^-\pi^+\bar\nu_\tau
$$

The detector therefore sees six charged tracks: one kaon and five pions. The two neutrinos are not reconstructed, so the signal does not form a narrow peak at the $B^0$ mass. *For this analysis, the three-prong $\tau$ decay provides a major experimental advantage that make the analysis measurement possible: the three pion tracks define a decay vertex clearly **detached** from the $B$ decay vertex. Vertex geometry can then be used to distinguish a real, finite-lived $\tau$ from backgrounds with the same visible particles.*

## What is vertex detachment, and how does it separate signal from prompt background?

In signal events, the $B^0$ first decays to a $D^{*-}$ and a $\tau^+$. The $\tau^+$ travels a measurable distance before producing the three pions, so the reconstructed $3\pi$ vertex should be downstream (ie. further along the detector in the beam direction) of and detached from the $B^0$ decay vertex. However, reconstruction of this signal decay is originally swamped by a prompt background $B\to D^*3\pi X$ (that will dominate the analysis and make signal extraction impossible if it is not selected out; also, note that $X$ here can mean no other particles, or potentially some additional particles that get missed), where the three pions are produced directly in the $B$ decay and their reconstructed vertex should therefore instead coincide with the $B$ vertex, within resolution.

![Schematic decay topologies for signal and prompt production](https://github.com/afernez/ML-BtoDstarTauNu-Tauto3Pi-background-removal/blob/master/signal_prompt_decay_diagrams.png?raw=true)

*Figure adapted from the analysis note. On the left, the signal $\tau$ travels away from the $B^0$ decay point before producing the three pions, giving two visibly separated vertices. On the right, the three pions are produced directly at the $B^0$ decay vertex, illustrating an example of prompt background.*

<details>
<summary><b>Advanced Aside on Prompt Backgrounds</b></summary>

For the interested reader: technically, it is entirely possible that the three pions come from a *different* $B$ decay than the $B$ that yields the $D^*$. We will also consider these backgrounds as part of the prompt category, although here the decay vertex yielding the three pions may be somewhat displaced from the chosen $B$ vertex.

</details>

The first iteration of this analysis (using Run 1 data from LHCb) selected out this prompt background by calculating a quantity related to the reconstructed $\tau$ flight distance that could reliably separate the signal from prompt background when it was cut on (recall "cutting" on a variable $a$ means you impose some requirement like $a > 100$ for all data). We want to continue to make use of this intelligent detachment quantity now, but *to get an even better result, it would be fantastic if we could incorporate more information about the events to separate the prompt background from the signal even more effectively: we want to see (as the Run 2 analysis successfully found) if some machine learning (ML) method can help us leverage this $\tau$ flight and other relevant information to accomplish this, without too much additional effort.*

## Why the $D_s^+$ peak matters

Prompt rejection is only one stage of the analysis. Once prompt $B\to D^*3\pi X$ events are reduced, **double-charm** decays—especially

$$
B\to D^{*-}D_s^+X,\qquad D_s^+\to\pi^+\pi^-\pi^+N,
$$

where $N$ denotes possible unreconstructed neutral particles—are the most important remaining background. These events are challenging precisely because a $D_s^+$ is also a finite-lived particle: its $3\pi$ vertex is genuinely detached from the $B$ vertex and can therefore look much more like a signal ($\tau$) decay than prompt pions do. Because this double charm background looks so similar to the signal, it is particularly difficult to remove. It is a fact of life that, if we want sufficient signal stats for the analysis, we will have to understand these double charm decays well.

To have any hope of achieving this understanding, it is important for the analysis to be able to understand the composition of these double charm decays. Recall from the intro activity that $D_s^+\to3\pi$ (with no additional particles missed in the reconstruction) decays produce a visible peak in the $m(3\pi)$ mass spectrum. Without going into too much detail here about exactly how this is useful, hopefully it is somewhat intuitive that seeing this $D_s^+$ mass peak as clearly as possible provides some handle for understanding these double charm decays. A strong detachment requirement can make the $D_s^+$ peak *cleaner*, suppressing the smooth prompt contribution underneath the peak while retaining displaced $D_s^+$ decays. The resulting peak can then be extremely valuable for studying and validating the double-charm background in data.

<details>
<summary><b>Aside: Why isn't the 3 pion mass enough to separate signal and double charm?</b></summary>

Signal decays should also peak roughly near the $\tau$ mass, although with a neutrino missing. This suggests that they can be separated from *some* of the double-charm background, but unfortunately not even close to all of it will be easily distinguished from signal using this mass alone. I invite you to consider how additional missed particles affect the $3\pi$ mass distribution.

</details>

<details>
<summary><b>Aside: How can double-charm background be separated from signal?</b></summary>

For the interested reader: separating double charm from signal requires investigating quantities such as neutral- and charged-particle isolation, the kinematics and different $3\pi$ resonance structures of $\tau$ and $D_s^+$ decays, and eventually the ability to fit different double-charm components in a dedicated control sample. These methods allow some of the double-charm background to be removed without sacrificing too much signal. For the remainder, they provide enough understanding of the full double-charm contribution to separate it confidently from signal in the analysis.

</details>

The goal of the present activity is therefore this: **develop some method to remove as much prompt $3\pi$ production as possible, and then ensure the presence of a clean $D_s^+$ component in data**.

In [ ]:
# Import the packages used throughout the activity
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
from scipy.optimize import curve_fit
from sklearn.metrics import auc, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Define a consistent random state for reproducible results
RANDOM_STATE = 42

# Initialize plot style options
style_name = "seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "seaborn-whitegrid"
plt.style.use(style_name)
plt.rcParams.update({"figure.figsize": (8, 5.5), "font.size": 12})

# Use the same sample colors throughout the activity.
SAMPLE_COLORS = {"Signal": "red", "Prompt": "grey", "Double charm": "black"}

# Download the activity datasets from Google Drive if they are not already cached
files_to_download = {
    "data_faked_ML.csv": "1IWrMsfZWgX87XfN6LmN4S6b6taY4Q-Yo",
    "data_faked.csv": "1viFX8ZF-NTaPJgpLNp3vzSaHQNPbAYog",
    "Prompt_start.csv": "1NDT0taTdHIQ6onRYNJfbqvQf_zgrZGaq",
    "Signal_start.csv": "1zjZKsBX9OfRTTo68ecGgWQpbAo6l33lx",
    "DoubleCharm_start.csv": "1c_n9KTgU2zbW4owhz6-NV5LbUG0JcRFO",
    "Prompt_ML.csv": "1HTxskMcIiAkIyVTHTuvi19HZJ4ImOenb",
    "Signal_ML.csv": "1CGiHiWqDgUw9LhrB65qs6XWUxyzd_3ti",
}

missing_files = [filename for filename in files_to_download if not Path(filename).is_file()]

if missing_files:
    import gdown

    for filename in missing_files:
        print(f"Downloading {filename} ...")
        result = gdown.download(id=files_to_download[filename], output=filename, quiet=False)
        if result is None or not Path(filename).is_file():
            raise RuntimeError(f"Download failed for {filename}")
else:
    print("All activity files are already cached locally.")

for filename in files_to_download:
    size_mb = Path(filename).stat().st_size / 1024**2
    print(f"{filename}: {size_mb:.1f} MB")

### Explore the available variables

The three `_start` files contain simulated **signal**, **prompt-background**, and **double-charm** events. The first cell below displays the available branch (column) names. A branch is simply one measured or calculated quantity stored for every event.

Choose one or more names by placing them in `VARIABLES_TO_PLOT` in the second cell, then rerun the cell. The code uses the same bin boundaries and normalizes each histogram to unit area, so we compare the **shapes** of the samples rather than their different numbers of simulated events. Look for variables for which the signal and prompt curves are well separated (note: I also am choosing to plot some double charm events here, but the red/signal and grey/prompt distributions are what you should be hoping to separate for now).

In [ ]:
START_FILES = {
    "Signal": Path("Signal_start.csv"),
    "Prompt": Path("Prompt_start.csv"),
    "Double charm": Path("DoubleCharm_start.csv"),
}

# Read only the header first: this shows our choices without loading the large files.
available_branches = pd.read_csv(START_FILES["Signal"], nrows=0).columns.tolist()
print("Available branches:")
for branch in available_branches:
    print(f"  {branch}")

In [ ]:
# Activity: replace or add branch names from the list printed above.
VARIABLES_TO_PLOT = [
    "pion1_E",
    "tau_decay_z",
]

unknown = [name for name in VARIABLES_TO_PLOT if name not in available_branches]
if unknown:
    raise ValueError(f"Unknown branch name(s): {unknown}")

# Loading only the chosen columns keeps this exploration reasonably fast and small.
start_samples = {
    label: pd.read_csv(filename, usecols=VARIABLES_TO_PLOT)
    for label, filename in START_FILES.items()
}

fig, axes = plt.subplots(
    len(VARIABLES_TO_PLOT), 1,
    figsize=(8, 4.5 * len(VARIABLES_TO_PLOT)),
    squeeze=False,
)

for variable, ax in zip(VARIABLES_TO_PLOT, axes[:, 0]):
    combined = pd.concat(
        [sample[variable] for sample in start_samples.values()],
        ignore_index=True,
    ).replace([np.inf, -np.inf], np.nan).dropna()
    low, high = combined.quantile([0.005, 0.995])
    bins = np.linspace(low, high, 51)

    for label, sample in start_samples.items():
        values = sample[variable].replace([np.inf, -np.inf], np.nan).dropna()
        ax.hist(
            values, bins=bins, density=True, histtype="step", linewidth=2,
            label=label, color=SAMPLE_COLORS[label],
        )

    ax.set_xlabel(variable)
    ax.set_ylabel("Normalized events")
    ax.legend()

fig.suptitle("Comparing simulated event samples", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()

### A first machine-learning attempt: **throw in the kitchen sink**

Several individual variables may show differences between signal and prompt events, but those differences can overlap and interact. A machine-learning model can examine many measurements together and learn combinations that are difficult to spot in one histogram at a time.

For this first attempt, we will deliberately avoid making clever physics-inspired variables. We will give each model all of the **low-level measurements**: the pion energy and momentum components, the reconstructed vertex positions, and their uncertainties. We will *not* yet give it calculated quantities such as `tau_delta_z_sig`, `tau_delta_r_sig`, `tauD_delta_z_sig`, any of the logarithmic variables, $m_{3\pi}$, or the compatibility and flight-angle variables. Without thinking too hard (like one needs to in order to define those variables in the first place), it's useful to see if we can throw in the kitchen sink to the ML classifiers and see if anything sticks. Later, we can think more carefully about which information is physically useful and how best to represent it.

We will train on simulated signal and prompt events. Twenty percent of the events will be held back as a **test sample** and never shown to either model during training. Both models return a number between 0 and 1: values near 1 mean “more signal-like,” while values near 0 mean “more prompt-like.”

Both methods have adjustable settings called **hyperparameters**, such as the number and size of trees or neural-network layers. For now, use the supplied choices; we will investigate hyperparameters more carefully later.

In [ ]:
# Select only directly measured/reconstructed low-level quantities.
LOW_LEVEL_FEATURES = [
    "pion1_E", "pion1_PX", "pion1_PY", "pion1_PZ",
    "pion2_E", "pion2_PX", "pion2_PY", "pion2_PZ",
    "pion3_E", "pion3_PX", "pion3_PY", "pion3_PZ",
    "tau_decay_x", "tau_decay_x_err",
    "tau_decay_y", "tau_decay_y_err",
    "tau_decay_z", "tau_decay_z_err",
    "tau_origin_x", "tau_origin_x_err",
    "tau_origin_y", "tau_origin_y_err",
    "tau_origin_z", "tau_origin_z_err",
    "D0_decay_x", "D0_decay_x_err",
    "D0_decay_y", "D0_decay_y_err",
    "D0_decay_z", "D0_decay_z_err",
]

signal_raw = pd.read_csv("Signal_start.csv", usecols=LOW_LEVEL_FEATURES)

# Use an equally sized prompt sample. This makes the classes balanced and keeps
# the activity quick enough to run on an ordinary laptop or hosted notebook.
prompt_raw = pd.read_csv("Prompt_start.csv", usecols=LOW_LEVEL_FEATURES)
prompt_raw = prompt_raw.sample(
    n=min(len(prompt_raw), len(signal_raw)),
    random_state=RANDOM_STATE,
)

signal_raw["label"] = 1
prompt_raw["label"] = 0
first_ml_data = pd.concat([signal_raw, prompt_raw], ignore_index=True)
first_ml_data[LOW_LEVEL_FEATURES] = first_ml_data[LOW_LEVEL_FEATURES].replace(
    [np.inf, -np.inf], np.nan
)

X_first_train, X_first_test, y_first_train, y_first_test = train_test_split(
    first_ml_data[LOW_LEVEL_FEATURES],
    first_ml_data["label"],
    test_size=0.20,
    stratify=first_ml_data["label"],
    random_state=RANDOM_STATE,
)

print(f"Low-level input variables: {len(LOW_LEVEL_FEATURES)}")
print(f"Training events: {len(X_first_train):,}")
print(f"Held-out test events: {len(X_first_test):,}")

### Boosted decision tree

A decision tree repeatedly divides events using questions such as “is this variable above or below some value?” A **boosted** decision tree builds many small trees in sequence, with later trees concentrating on events that earlier trees handled poorly. Their answers are added together to produce the final signal-likeness score. XGBoost is a widely used, efficient implementation. The settings below control quantities such as the number of trees, their depth, and how strongly each new tree changes the answer.

![Diagram showing successive decision trees combining into a signal-likeness score](https://github.com/afernez/ML-BtoDstarTauNu-Tauto3Pi-background-removal/blob/master/bdt_boosting_diagram.svg?raw=true)

In [ ]:
first_xgb = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.08,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=4,
)
first_xgb.fit(X_first_train, y_first_train)
xgb_test_score = first_xgb.predict_proba(X_first_test)[:, 1]

score_bins = np.linspace(0, 1, 41)
plt.hist(xgb_test_score[y_first_test.to_numpy() == 1], bins=score_bins,
         density=True, histtype="step", linewidth=2, label="Signal",
         color=SAMPLE_COLORS["Signal"])
plt.hist(xgb_test_score[y_first_test.to_numpy() == 0], bins=score_bins,
         density=True, histtype="step", linewidth=2, label="Prompt",
         color=SAMPLE_COLORS["Prompt"])
plt.xlabel("XGBoost signal-likeness score")
plt.ylabel("Normalized events")
plt.title("XGBoost scores for events not used in training")
plt.legend()
plt.show()

### Neural network

A neural network passes the input variables through layers of simple mathematical units (weights and biases). They generally have multiple layers of nodes (with associated biases) connected by edges (with associated weights), usually plus activation functions, that all transform the data. The first layer is the input layer, followed by hidden layers that will be trained to transform the inputs in a meaningful way such that the final layer, the output layer (here just one node—**do you understand why?**), can accurately classify how signal-like a given event is.

![Diagram showing input variables passing through neural-network layers to a signal-likeness score](https://github.com/afernez/ML-BtoDstarTauNu-Tauto3Pi-background-removal/blob/master/neural_network_diagram.svg?raw=true)


<details>
<summary><b>Technical notes for implementing neural nets</b></summary>

Neural networks are sensitive to the numerical scales of their inputs, so we first replace missing values and standardize every variable to have roughly zero mean and unit spread. The final one-unit `sigmoid` layer guarantees an output between 0 and 1.

</details>

In [ ]:
# Learn all preprocessing choices from the training sample only.
first_imputer = SimpleImputer(strategy="median")
first_scaler = StandardScaler()
X_nn_train = first_imputer.fit_transform(X_first_train)
X_nn_train = first_scaler.fit_transform(X_nn_train).astype("float32")
X_nn_test = first_imputer.transform(X_first_test)
X_nn_test = first_scaler.transform(X_nn_test).astype("float32")

tf.keras.utils.set_random_seed(RANDOM_STATE)
first_nn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(LOW_LEVEL_FEATURES),)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
first_nn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
)
first_nn.fit(
    X_nn_train,
    y_first_train.to_numpy(),
    validation_split=0.20,
    epochs=20,
    batch_size=1024,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )],
    verbose=0,
)
nn_test_score = first_nn.predict(X_nn_test, batch_size=2048, verbose=0).ravel()

plt.hist(nn_test_score[y_first_test.to_numpy() == 1], bins=score_bins,
         density=True, histtype="step", linewidth=2, label="Signal",
         color=SAMPLE_COLORS["Signal"])
plt.hist(nn_test_score[y_first_test.to_numpy() == 0], bins=score_bins,
         density=True, histtype="step", linewidth=2, label="Prompt",
         color=SAMPLE_COLORS["Prompt"])
plt.xlabel("Neural-network signal-likeness score")
plt.ylabel("Normalized events")
plt.title("Neural-network scores for events not used in training")
plt.legend()
plt.show()

Interestingly, the neural net seems to do better than the XGBoost method!

<details>
<summary><b>Can we draw some general conclusions from this?</b></summary>

A plausible explanation here is that we supplied only raw coordinates, uncertainties, energies, and momenta. Useful physics quantities often involve smooth combinations of several of these measurements: differences between vertex positions, divisions by uncertainties, angles, distances, and correlations among momentum components. A neural network can build smooth combinations of many inputs through its connected layers, whereas a collection of finite-depth decision trees approximates them using many rectangular, “greater than or less than” regions. With these particular settings and raw inputs, that may make the useful relationships easier for the neural network to learn. This comparison is only a first look, however. Different hyperparameters could change the result, and a score plot alone does not establish that one type of model is universally superior.

</details>


So far, we have charged ahead with machine learning and asked the models to discover everything from low-level measurements. We can do better by borrowing knowledge from the physicists who designed the analysis. They know which geometrical relationships should distinguish a real, displaced $\tau$ decay from three pions produced promptly at the $B$ vertex. We will now inspect variables that encode those relationships directly, then see what happens when the ML models are trained on this more purposeful description of each event.

## Using the decay geometry: physics-informed variables

Our first ML models received the low-level coordinates, momenta, and uncertainties and had to discover useful relationships for themselves. Particle physicists can instead use the expected decay geometry to construct variables that present those relationships more directly. The following quantities define the six inputs used by the **detachment BDT** in the actual Run 2 $R(D^*)$ analysis. Here, “$\tau$ vertex” means the reconstructed vertex of the three-pion candidate—even when the candidate is actually prompt background and no real $\tau$ is present.

Note that, for the first two quantities, after requiring them to be positive, the analysis trains with `log_tau_delta_z_sig` and `log_tau_delta_r_sig`, their logarithms, to compress their long positive tails.

- **`tau_delta_z_sig`** is the signed separation along the beam direction between the $\tau$ decay vertex and its origin, the $B$ decay vertex, divided by the combined uncertainty:

  $$\frac{z(\tau)-z(B)}{\sqrt{\sigma^2_{z(\tau)}+\sigma^2_{z(B)}}}.$$

  A real $\tau$ travels downstream before decaying, so signal should usually have a convincingly positive value. Prompt pions originate at the $B$ vertex, so their separation should be compatible with zero apart from resolution and reconstruction effects.

- **`tau_delta_r_sig`** is the corresponding signed change in radial distance from the beam line, $[r(\tau)-r(B)]$, divided by its propagated uncertainty, where $r=\sqrt{x^2+y^2}$. A positive value means that the $\tau$ vertex lies farther from the beam line than its origin, as expected for a particle flying outward before decaying.

- **`tauD_delta_z_sig`** compares the $\bar D^0$ and $\tau$ decay vertices along the beam direction:

  $$\frac{z(\bar D^0)-z(\tau)}{\sqrt{\sigma^2_{z(\bar D^0)}+\sigma^2_{z(\tau)}}}.$$

  The $\bar D^0$ and $\tau$ both descend from the same $B$ decay chain but have their own finite flights. Their relative vertex positions therefore contain useful topology information that is difficult to express with either vertex alone.

- **`log_tau_flight_angle`** is $\log[\arccos(\mathrm{DIRA})]$, where DIRA is the cosine of the angle between the reconstructed $\tau$ momentum and the line from its origin to its decay vertex. A genuine flying particle should point approximately along its line of flight, giving DIRA near 1 and a small angle. Taking the logarithm spreads out these very small angles so a model can use them more easily.

- **`log_tau_decay_compat`** is the logarithm of the $\chi^2$ (note: if you aren't familiar with $\chi^2$ from statistics, it's no problem here) of the three-track $\tau$ decay-vertex fit (another fit... again, don't worry too much about what this means!). It measures how compatible the three pion tracks are with a common decay point.

- **`log_tau_origin_compat`** is the logarithm of the $\chi^2$ associated with the reconstructed origin of the $\tau$ candidate. It measures how geometrically compatible the candidate is with that proposed production vertex.

The logarithms compress variables that cover very wide numerical ranges while preserving their ordering. Collectively, these six variables describe **how far the candidate flew, whether it flew in a physically sensible direction, and how well its production and decay vertices were reconstructed**.

### Loose detachment requirements before ML training

Before training the detachment classifiers again, let's make some use of these new variables. In particular, we'll follow the $R(D^*)$ analysis' choice and look at the effect of some loose (we'll see below how loose they are!) cuts on separating signal and prompt background:

```
tau_delta_z_sig>2, tau_delta_r_sig>0
```

The first requirement asks that the downstream displacement be more than two combined standard deviations from zero. The second asks only that the candidate move outward from the beam line. These are deliberately loose cuts: they should keep most genuinely displaced signal and double-charm decays while removing a substantial fraction of prompt candidates before the machine-learning model is trained.

Let us check that statement directly using the three `_start` samples. The shaded regions in the plots are rejected by the corresponding requirement.

In [ ]:
CUT_VARIABLES = ["tau_delta_z_sig", "tau_delta_r_sig"]
cut_samples = {
    label: pd.read_csv(filename, usecols=CUT_VARIABLES)
    for label, filename in START_FILES.items()
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_settings = {
    "tau_delta_z_sig": {
        "range": (-5, 25), "cut": 2.0,
        "xlabel": r"Longitudinal displacement significance, $\Delta z/\sigma_{\Delta z}$",
    },
    "tau_delta_r_sig": {
        "range": (-10, 25), "cut": 0.0,
        "xlabel": r"Radial displacement significance, $\Delta r/\sigma_{\Delta r}$",
    },
}

for ax, variable in zip(axes, CUT_VARIABLES):
    settings = plot_settings[variable]
    bins = np.linspace(*settings["range"], 61)
    for label, sample in cut_samples.items():
        values = sample[variable].replace([np.inf, -np.inf], np.nan).dropna()
        ax.hist(values, bins=bins, density=True, histtype="step", linewidth=2,
                label=label, color=SAMPLE_COLORS[label])
    ax.axvspan(settings["range"][0], settings["cut"], color="gold", alpha=0.12,
               label="Rejected region")
    ax.axvline(settings["cut"], color="darkgoldenrod", linestyle="--", linewidth=2)
    ax.set_xlabel(settings["xlabel"])
    ax.set_ylabel("Normalized events")
    ax.set_xlim(settings["range"])

axes[0].legend()
fig.suptitle("Loose physics-motivated detachment requirements", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

The signal and double-charm distributions are shifted toward positive displacement, whereas prompt events accumulate much closer to zero and extend into the unphysical negative region because of finite resolution and incorrect combinations.

Try changing `TRY_Z_CUT` and `TRY_R_CUT` below and rerunning the cell. Can you reject more prompt background without giving up too much signal? There is no single universally “best” cut: tightening a requirement always trades signal efficiency for background rejection. After experimenting, restore the analysis values `2.0` and `0.0` before continuing.

In [ ]:
# Experiment with these values, then restore 2.0 and 0.0 before continuing.
TRY_Z_CUT = 2.0
TRY_R_CUT = 0.0

cut_results = []
for label, sample in cut_samples.items():
    passes = (
        (sample["tau_delta_z_sig"] > TRY_Z_CUT)
        & (sample["tau_delta_r_sig"] > TRY_R_CUT)
    )
    retained = 100 * passes.mean()
    cut_results.append({
        "Sample": label,
        "Events before": len(sample),
        "Events after": int(passes.sum()),
        "Retained (%)": retained,
        "Rejected (%)": 100 - retained,
    })

cut_results = pd.DataFrame(cut_results).set_index("Sample")
display(cut_results.style.format({
    "Events before": "{:,}",
    "Events after": "{:,}",
    "Retained (%)": "{:.1f}",
    "Rejected (%)": "{:.1f}",
}))

With the analysis choices, we retain roughly **84% of signal** while rejecting roughly **59% of prompt background** in these samples. Double charm mostly survives because its $3\pi$ vertex often comes from a genuinely displaced charm-hadron decay—the same reason it remains an important background later.

From this point onward, we use the `_ML.csv` files. They were prepared using the fixed requirements

```python
tau_delta_z_sig > 2.0
tau_delta_r_sig > 0.0
```

together with a few more purity selections. Thus, the next models begin with samples that are already mildly enriched in displaced candidates; their task is to use the six physics-informed variables to obtain better separation than these two loose cuts alone.

**Check your understanding: do you expect that cutting out prompt background before beginning training will make it more difficult for the ML methods to achieve the same level of performance as before? Would similar ML performance now actually indicate better combined (cut + ML) signal and prompt separation power? Do you expect these new calculated variables will add much information to our ML methods? Which ML methods do you think the new input variables will affect more: XGBoost or neural nets?**

## Load the physics-informed training samples

We now load the `_ML.csv` signal and prompt samples produced with the loose detachment requirements above. The models in this section begin with only the six physics-informed detachment variables.

In [ ]:
# Define the files and the six physics-informed variables.
SIGNAL_FILE = Path("Signal_ML.csv")
PROMPT_FILE = Path("Prompt_ML.csv")

FEATURES = [
    "log_tau_delta_z_sig",
    "log_tau_delta_r_sig",
    "tauD_delta_z_sig",
    "log_tau_flight_angle",
    "log_tau_decay_compat",
    "log_tau_origin_compat",
]

# Read the six physics-informed variables.
signal = pd.read_csv(SIGNAL_FILE, usecols=FEATURES)
prompt = pd.read_csv(PROMPT_FILE, usecols=FEATURES)
signal["label"] = 1
prompt["label"] = 0
data = pd.concat([signal, prompt], ignore_index=True)

# Print the sample sizes.
print(f"Signal events: {len(signal):,}")
print(f"Prompt events: {len(prompt):,}")

## Inspect the six physics-informed inputs

The `_ML` files already contain finite, analysis-ready values, so we can move directly to comparing their distributions. Before training either model, study the shapes below. **Which variables do you predict will be most important for separating signal from prompt background?**

Remember that these samples have already passed `tau_delta_z_sig > 2` and `tau_delta_r_sig > 0`. Consequently, the first two plots begin at the logarithms of those thresholds rather than showing the rejected prompt-dominated regions from the previous section.

In [ ]:
# Plot all six variables with shared bins for a fair shape comparison.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for feature, ax in zip(FEATURES, axes.flat):
    combined = data[feature]
    lo, hi = combined.quantile([0.005, 0.995])
    bins = np.linspace(lo, hi, 50)
    ax.hist(signal[feature], bins=bins, density=True, histtype="step", lw=2,
            label="Signal", color=SAMPLE_COLORS["Signal"])
    ax.hist(prompt[feature], bins=bins, density=True, histtype="step", lw=2,
            label="Prompt", color=SAMPLE_COLORS["Prompt"])
    ax.set_xlabel(feature)
    ax.set_ylabel("Normalized events")
axes.flat[0].legend()
fig.suptitle("Detachment-BDT input variables", y=1.02, fontsize=16)
fig.tight_layout()
plt.show()

## Create independent training, validation, and test samples

The split is stratified so each subset has the same signal-to-prompt ratio. The validation subset controls early stopping and helps us choose hyperparameters. The test subset is never used to fit either model; its score plots show how each trained model behaves on unseen events.

Because the prompt sample is much larger, `scale_pos_weight` gives signal and prompt equal total importance for XGBoost. We use the same class-count ratio as a training weight for the neural network.

In [ ]:
X = data[FEATURES]
y = data["label"]

#Split off Testing data from the Training and Validation Data
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

#Split the Validation and the Training Data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25,
    stratify=y_train_val, random_state=RANDOM_STATE
)

#Calculate how many prompt vs signal events in the training data
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Train/validation/test sizes: {len(X_train):,} / {len(X_val):,} / {len(X_test):,}")
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

## Explore and tune both models

### XGBoost hyperparameters: what do the knobs control?

A boosted decision tree is built sequentially. Each new tree tries to correct mistakes made by the trees already in the ensemble. **Hyperparameters** are choices we make about that learning process; unlike the tree splits and leaf values, they are not learned directly by the fit.

The parameters used in this activity are:

| Parameter | What it controls | What happens when it is increased? | Reasonable values to try |
|---|---|---|---|
| `n_estimators` | Maximum number of sequential trees | More opportunities to learn, but longer training and eventually overfitting | 100–1200 with early stopping |
| `learning_rate` | Shrinkage applied to each new tree | Each tree makes a larger correction; training is faster but can become less stable | 0.01–0.3 |
| `max_depth` | Maximum number of splits from a tree root to a leaf | Trees can model more complicated feature interactions, but overfit more easily | 2–8 |
| `min_child_weight` | Minimum total event weight needed to create a child node | Prevents leaves supported by only a small effective sample; larger values make smoother, more conservative trees | 1–30 |
| `subsample` | Fraction of training events randomly offered to each tree | Values below 1 add randomness and reduce overfitting, but too small a value throws away useful information | 0.5–1.0 |
| `colsample_bytree` | Fraction of input variables offered to each tree | Values below 1 decorrelate trees; with only six inputs, very small values may hide important physics | 0.5–1.0 |
| `gamma` | Minimum loss improvement required to make an additional split | More weak splits are rejected, producing simpler trees | 0–2 |
| `reg_alpha` | $L_1$ penalty on leaf weights | Encourages some leaf corrections to become exactly zero; larger values simplify the model | 0–2 |
| `reg_lambda` | $L_2$ penalty on leaf weights | Smoothly shrinks extreme leaf predictions; larger values make the model more conservative | 0.1–20 |
| `scale_pos_weight` | Relative weight assigned to signal events | Compensates for the much larger prompt sample; here it is calculated as $N_{\rm prompt}/N_{\rm signal}$ | Usually the class-count ratio |
| `early_stopping_rounds` | Number of rounds without validation improvement allowed before stopping | Larger values are more patient but take longer and may chase statistical fluctuations | 20–100 |

> ### Important note: ROC AUC performance metric
>
> **ROC AUC** means the **area under the receiver operating characteristic curve** (we will look at some example variants of these ROC curves later in this notebook). It summarizes how well a classifier ranks signal above prompt background across *all possible score cuts*. Equivalently, it is the probability that a randomly selected signal event receives a higher score than a randomly selected prompt event. An ROC AUC of **0.5** is no better than random ranking; **1.0** is perfect ranking. Larger is better.

A few parameters are mostly computational choices here. `objective="binary:logistic"` asks for a binary score between 0 and 1, while `eval_metric="auc"` asks XGBoost to monitor the ROC AUC defined above. `tree_method="hist"` groups continuous values into bins to train quickly, while `max_bin` controls the number of such bins. `random_state` makes stochastic choices reproducible, and `n_jobs` sets CPU parallelism.

Parameters interact. In particular, a smaller `learning_rate` normally needs more trees, and deeper trees generally need stronger regularization. The goal is not simply to maximize training AUC: a large gap between training and validation performance is evidence of overfitting.

### Neural-network hyperparameters

The neural network has a different set of controls:

| Parameter | What it controls | What happens when it is increased? | Reasonable values to try |
|---|---|---|---|
| `hidden_layers` | Number of nodes in each hidden layer | More nodes/layers can learn more complicated relationships, but take longer and can overfit | `(32, 16)`, `(64, 32)`, `(128, 64)` |
| `dropout` | Fraction of hidden-node outputs randomly hidden during each training step | Adds regularization; too much prevents the model from learning | 0–0.3 |
| `learning_rate` | Size of each optimizer update | Learns faster, but overly large steps may skip over a good solution | 0.0003–0.01 |
| `batch_size` | Events used to estimate each update | Larger batches are faster and smoother but make fewer updates per epoch | 256–4096 |
| `epochs` | Maximum passes through the training data | Gives more time to learn but eventually risks overfitting | 10–50 with early stopping |
| `patience` | Epochs without validation improvement before stopping | Allows more time to recover from fluctuations, at the cost of longer training | 2–8 |

We standardize every neural-network input using the training sample only. This prevents a variable measured in tens of thousands from dominating another variable measured near one simply because of its units. As with XGBoost, changing one or two hyperparameters at a time makes their effects much easier to interpret.

### Your turn: choose trial configurations

Edit `student_xgb_parameters` and `student_nn_parameters` below, then rerun the cell. Start by changing one or two related parameters at a time so you can understand their effect. The plots always use events that were held out from fitting, with signal in red and prompt in grey. Look for both strong separation and sensible, smooth score shapes.

Some useful questions to investigate (don't worry too much if you don't think you'll have time to investigate all of them!):

1. What happens when the trees are made much deeper?
2. If you reduce `learning_rate`, how must `n_estimators` change?
3. Can `min_child_weight` or `reg_lambda` recover performance from an over-complex XGBoost model?
4. Does adding neural-network nodes always help? What happens with no dropout or very large dropout?
5. How do `learning_rate`, `batch_size`, and the number of training epochs interact?

The models use the training sample for fitting and validation for early stopping. The score plots use the held-out test events requested for this activity. Strictly speaking, repeatedly choosing settings because their test plot looks best makes that test sample part of the decision process; a publication-quality analysis would reserve yet another untouched sample for its final quoted performance.

In [ ]:
# Edit these two dictionaries to test your own hypotheses.
student_xgb_parameters = dict(
    n_estimators=600,
    learning_rate=0.08,
    max_depth=3,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=1.0,
    gamma=0.0,
    reg_alpha=0.0,
    reg_lambda=2.0,
)
student_nn_parameters = dict(
    hidden_layers=(64, 32),
    dropout=0.10,
    learning_rate=1e-3,
    batch_size=2048,
    epochs=25,
    patience=3,
)

def make_xgb(parameters, early_stopping_rounds=40):
    options = dict(
        objective="binary:logistic", eval_metric="auc", tree_method="hist",
        max_bin=256, scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, n_jobs=4, **parameters,
    )
    fit_options = dict(eval_set=[(X_val, y_val)], verbose=False)
    if int(xgb.__version__.split(".")[0]) >= 2:
        options["early_stopping_rounds"] = early_stopping_rounds
    else:
        fit_options["early_stopping_rounds"] = early_stopping_rounds
    return xgb.XGBClassifier(**options), fit_options

def fit_neural_network(X_train_input, X_val_input, X_test_input, parameters):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(X_train_input).astype("float32")
    val_scaled = scaler.transform(X_val_input).astype("float32")
    test_scaled = scaler.transform(X_test_input).astype("float32")

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    layers = [tf.keras.layers.Input(shape=(X_train_input.shape[1],))]
    for width in parameters["hidden_layers"]:
        layers.extend([
            tf.keras.layers.Dense(width, activation="relu"),
            tf.keras.layers.Dropout(parameters["dropout"]),
        ])
    layers.append(tf.keras.layers.Dense(1, activation="sigmoid"))
    network = tf.keras.Sequential(layers)
    network.compile(
        optimizer=tf.keras.optimizers.Adam(parameters["learning_rate"]),
        loss="binary_crossentropy",
    )
    history = network.fit(
        train_scaled, y_train.to_numpy(),
        validation_data=(val_scaled, y_val.to_numpy()),
        epochs=parameters["epochs"], batch_size=parameters["batch_size"],
        class_weight={0: 1.0, 1: scale_pos_weight},
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=parameters["patience"],
            restore_best_weights=True,
        )],
        verbose=0,
    )
    test_scores = network.predict(test_scaled, batch_size=4096, verbose=0).ravel()
    return network, scaler, history, test_scores

# Train the trial XGBoost model.
student_xgb, student_xgb_fit = make_xgb(student_xgb_parameters, 30)
student_xgb.fit(X_train, y_train, **student_xgb_fit)
student_xgb_scores = student_xgb.predict_proba(X_test)[:, 1]

# Train the trial neural network.
student_nn, student_nn_scaler, student_nn_history, student_nn_scores = fit_neural_network(
    X_train, X_val, X_test, student_nn_parameters
)

print(f"XGBoost validation AUC: {student_xgb.best_score:.5f}")
print(f"Neural-network test AUC: {roc_auc_score(y_test, student_nn_scores):.5f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scores, title in [
    (axes[0], student_xgb_scores, "Trial XGBoost"),
    (axes[1], student_nn_scores, "Trial neural network"),
]:
    ax.hist(scores[y_test.to_numpy() == 1], bins=np.linspace(0, 1, 41), density=True,
            histtype="step", linewidth=2, color=SAMPLE_COLORS["Signal"], label="Signal")
    ax.hist(scores[y_test.to_numpy() == 0], bins=np.linspace(0, 1, 41), density=True,
            histtype="step", linewidth=2, color=SAMPLE_COLORS["Prompt"], label="Prompt")
    ax.set(xlabel="Signal-likeness score", ylabel="Normalized events", title=title)
    ax.legend()
fig.tight_layout()
plt.show()

### Tuned reference configurations

You have now had a chance to experiment with the model controls. For the rest of the activity, we will use the settings below as **likely near-optimal hyperparameters**. They provide a strong, fair reference point for comparing XGBoost with the neural network; you do not need to understand every numerical choice before continuing.

<details>
<summary><b>How were these settings chosen? (optional technical detail)</b></summary>

The values below were selected with a reproducible 24-point randomized search using validation ROC AUC only. The search varied tree depth, learning rate, minimum child weight, row and column subsampling, split penalty, and $L_1/L_2$ regularization. The held-out test sample was not examined until after one configuration had been selected.

The chosen XGBoost configuration uses a small learning rate and moderately deep trees, while a large minimum child weight, row subsampling, and stronger $L_2$ regularization control overfitting. Up to 1200 boosting rounds are available, but early stopping selects the useful number from the validation sample.

For the neural network, a small comparison of layer widths found that `(128, 64)` performed slightly better than `(32, 16)`, `(64, 32)`, and `(64, 32, 16)`. A 15% dropout rate provides light regularization. These are reasonable bounded optimizations for the activity rather than claims of globally optimal classifiers.

</details>

In [ ]:
# Reference hyperparameters selected using validation performance.
optimal_xgb_parameters = dict(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=20,
    subsample=0.75,
    colsample_bytree=1.0,
    gamma=0.0,
    reg_alpha=0.01,
    reg_lambda=5.0,
)
optimal_nn_parameters = dict(
    hidden_layers=(128, 64),
    dropout=0.15,
    learning_rate=1e-3,
    batch_size=2048,
    epochs=30,
    patience=4,
)

tuned_xgb, tuned_xgb_fit = make_xgb(optimal_xgb_parameters, 50)
tuned_xgb.fit(X_train, y_train, **tuned_xgb_fit)
xgb_test_score = tuned_xgb.predict_proba(X_test)[:, 1]

tuned_nn, tuned_nn_scaler, tuned_nn_history, nn_test_score = fit_neural_network(
    X_train, X_val, X_test, optimal_nn_parameters
)

# Keep the name `model` for the later XGBoost application-to-data section.
model = tuned_xgb
test_score = xgb_test_score
print(f"XGBoost best round: {tuned_xgb.best_iteration + 1}")
print(f"XGBoost test AUC: {roc_auc_score(y_test, xgb_test_score):.5f}")
print(f"Neural-network epochs: {len(tuned_nn_history.history['loss'])}")
print(f"Neural-network test AUC: {roc_auc_score(y_test, nn_test_score):.5f}")

In [ ]:
# Compare the tuned models on the same held-out test events.
score_bins = np.linspace(0, 1, 51)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scores, title in [
    (axes[0], xgb_test_score, "Tuned XGBoost"),
    (axes[1], nn_test_score, "Tuned neural network"),
]:
    ax.hist(scores[y_test.to_numpy() == 1], bins=score_bins, density=True,
            histtype="step", linewidth=2, color=SAMPLE_COLORS["Signal"], label="Signal")
    ax.hist(scores[y_test.to_numpy() == 0], bins=score_bins, density=True,
            histtype="step", linewidth=2, color=SAMPLE_COLORS["Prompt"], label="Prompt")
    ax.set(xlabel="Signal-likeness score", ylabel="Normalized events", title=title)
    ax.legend()
fig.suptitle("Physics-informed models on held-out test events", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

## What happens when we add genuine noise?

Real datasets often contain variables that happen to be available but have no useful relationship to the classification target. Giving a model more inputs therefore does not automatically give it more information. To test this directly, we will create six artificial columns drawn from Gaussian distributions. Each column has its own mean and width, but—crucially—the **same distribution is used for signal and prompt**, so none of these variables contains real separation power.

**Before running the next cell, make a prediction:** will the random inputs make either model perform worse by distracting it or encouraging overfitting? Will regularization and early stopping allow the model to ignore them? Could XGBoost and the neural network respond differently?

For a controlled comparison, we keep exactly the same events, split, and tuned hyperparameters. The only change is whether the six random columns are included. `RANDOM_STATE` makes the exercise reproducible.

In [ ]:
# Generate six random columns. Different columns have different scales, but
# signal and prompt are drawn from exactly the same distributions.
NOISE_FEATURES = [f"random_noise_{i}" for i in range(1, 7)]
noise_means = [-10.0, -2.0, 0.0, 3.0, 20.0, 100.0]
noise_widths = [0.5, 5.0, 1.0, 0.2, 10.0, 50.0]
rng = np.random.default_rng(RANDOM_STATE)
for feature, mean, width in zip(NOISE_FEATURES, noise_means, noise_widths):
    data[feature] = rng.normal(loc=mean, scale=width, size=len(data))

AUGMENTED_FEATURES = FEATURES + NOISE_FEATURES
X_aug_train = data.loc[X_train.index, AUGMENTED_FEATURES]
X_aug_val = data.loc[X_val.index, AUGMENTED_FEATURES]
X_aug_test = data.loc[X_test.index, AUGMENTED_FEATURES]

augmented_xgb, augmented_xgb_fit = make_xgb(optimal_xgb_parameters, 50)
augmented_xgb_fit["eval_set"] = [(X_aug_val, y_val)]
augmented_xgb.fit(X_aug_train, y_train, **augmented_xgb_fit)
augmented_xgb_score = augmented_xgb.predict_proba(X_aug_test)[:, 1]

augmented_nn, augmented_nn_scaler, augmented_nn_history, augmented_nn_score = fit_neural_network(
    X_aug_train, X_aug_val, X_aug_test, optimal_nn_parameters
)

feature_comparison = pd.DataFrame({
    "Model": ["XGBoost", "XGBoost", "Neural network", "Neural network"],
    "Inputs": ["6 physics variables", "6 physics + 6 random variables",
               "6 physics variables", "6 physics + 6 random variables"],
    "Test ROC AUC": [
        roc_auc_score(y_test, xgb_test_score),
        roc_auc_score(y_test, augmented_xgb_score),
        roc_auc_score(y_test, nn_test_score),
        roc_auc_score(y_test, augmented_nn_score),
    ],
})
display(feature_comparison.style.format({"Test ROC AUC": "{:.4f}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, scores, title in [
    (axes[0], augmented_xgb_score, "XGBoost with random inputs"),
    (axes[1], augmented_nn_score, "Neural network with random inputs"),
]:
    ax.hist(scores[y_test.to_numpy() == 1], bins=score_bins, density=True,
            histtype="step", linewidth=2, color=SAMPLE_COLORS["Signal"], label="Signal")
    ax.hist(scores[y_test.to_numpy() == 0], bins=score_bins, density=True,
            histtype="step", linewidth=2, color=SAMPLE_COLORS["Prompt"], label="Prompt")
    ax.set(xlabel="Signal-likeness score", ylabel="Normalized events", title=title)
    ax.legend()
fig.suptitle("Adding six variables containing no class information", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

# Restore convenient names for the six-variable XGBoost model used below.
signal_scores = xgb_test_score[y_test.to_numpy() == 1]
prompt_scores = xgb_test_score[y_test.to_numpy() == 0]

### What actually happened?

The random variables slightly **decrease** the held-out performance of both methods. In this run, the XGBoost ROC AUC changes from about 0.7862 to 0.7849, while the neural-network AUC changes from about 0.7872 to 0.7850. Because the generated variables have identical distributions for signal and prompt, any apparent pattern involving them is only a statistical fluctuation in the finite training sample—not information that should generalize to new events.

The damage is small rather than catastrophic. Regularization, validation-based early stopping, and the large training sample allow both models to ignore most of the noise. Nevertheless, the extra inputs give the models more opportunities to learn accidental patterns, making training a little more difficult and producing a small loss in generalization. Additionally, adding the random noise inputs also made the training for both models slower, although I don't report the training times here (because it's still small in either case; but if you're looking to scale up your data/input features, this issue can become more relevant).

This is why analysts do not simply include every available branch. Each additional input should have a plausible motivation when being added into the training of the model.

# Final evaluation of the physics-informed classifiers

We now return to the best-found XGBoost and neural-network models trained using only the six physics-motivated variables. A possible **bonus activity** is to add selected low-level quantities back one group at a time and repeat the controlled comparison. In quick tests, adding some low-level variables produced small improvements, but nothing dramatic: the six engineered quantities already capture most of the useful detachment information.

## Signal acceptance versus prompt rejection

For a selection `signal_score > cut`, signal acceptance is the fraction of signal retained and prompt rejection is one minus the fraction of prompt events retained. We compare both multivariate models with progressively tighter cuts on `tau_delta_z_sig`, similar to the simpler strategy used in the Run 1 analysis. All curves use the same held-out test events.

In [ ]:
# ROC curves for the two best six-variable models.
xgb_fpr, xgb_acceptance, xgb_thresholds = roc_curve(y_test, xgb_test_score)
nn_fpr, nn_acceptance, nn_thresholds = roc_curve(y_test, nn_test_score)
xgb_rejection = 1.0 - xgb_fpr
nn_rejection = 1.0 - nn_fpr
xgb_auc = roc_auc_score(y_test, xgb_test_score)
nn_auc = roc_auc_score(y_test, nn_test_score)

# Recover the untransformed significance and evaluate tighter one-variable cuts.
tau_delta_z_sig_test = np.exp(X_test["log_tau_delta_z_sig"])
signal_z = tau_delta_z_sig_test.loc[y_test == 1]
prompt_z = tau_delta_z_sig_test.loc[y_test == 0]
z_cut_values = np.array([2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 15.0, 20.0])
z_acceptance = np.array([(signal_z > cut).mean() for cut in z_cut_values])
z_rejection = np.array([1.0 - (prompt_z > cut).mean() for cut in z_cut_values])

xgb_signal_scores = xgb_test_score[y_test.to_numpy() == 1]
xgb_prompt_scores = xgb_test_score[y_test.to_numpy() == 0]
nn_signal_scores = nn_test_score[y_test.to_numpy() == 1]
nn_prompt_scores = nn_test_score[y_test.to_numpy() == 0]

# Find score cuts corresponding to several useful target signal acceptances.
target_acceptances = np.array([0.80, 0.65, 0.50, 0.35, 0.20])

def score_working_points(signal_scores, prompt_scores, targets):
    cuts = np.quantile(signal_scores, 1.0 - targets)
    acceptance = np.array([(signal_scores > cut).mean() for cut in cuts])
    rejection = np.array([1.0 - (prompt_scores > cut).mean() for cut in cuts])
    return cuts, acceptance, rejection

xgb_reference_cuts, xgb_reference_acceptance, xgb_reference_rejection = score_working_points(
    xgb_signal_scores, xgb_prompt_scores, target_acceptances
)
nn_reference_cuts, nn_reference_acceptance, nn_reference_rejection = score_working_points(
    nn_signal_scores, nn_prompt_scores, target_acceptances
)

# Keep the 50%-signal quantities for the model-saving and mass-spectrum sections.
half_index = int(np.where(np.isclose(target_acceptances, 0.50))[0][0])
xgb_cut_50 = float(xgb_reference_cuts[half_index])
nn_cut_50 = float(nn_reference_cuts[half_index])
xgb_acceptance_50 = float(xgb_reference_acceptance[half_index])
nn_acceptance_50 = float(nn_reference_acceptance[half_index])
xgb_rejection_50 = float(xgb_reference_rejection[half_index])
nn_rejection_50 = float(nn_reference_rejection[half_index])

fig, ax = plt.subplots(figsize=(9, 7.5))
ax.plot(xgb_acceptance, xgb_rejection, lw=2.5, color="#2166ac",
        label=f"XGBoost (AUC = {xgb_auc:.4f})")
ax.plot(nn_acceptance, nn_rejection, lw=2.5, linestyle="--", color="#7b3294",
        label=f"Neural network (AUC = {nn_auc:.4f})")
ax.plot(z_acceptance, z_rejection, marker="s", ms=5, lw=2,
        color="#e6a700", label=r"Tighter cuts on $\Delta z/\sigma_{\Delta z}$")
for cut, x, y_point in zip(z_cut_values, z_acceptance, z_rejection):
    ax.annotate(f">{cut:g}", (x, y_point), xytext=(4, 5),
                textcoords="offset points", fontsize=8, color="#9a6f00")

# Label the classifier score cuts at 80%, 65%, 50%, 35%, and 20% signal acceptance.
ax.scatter(xgb_reference_acceptance, xgb_reference_rejection, s=50, marker="o",
           color="#2166ac", edgecolor="white", zorder=4)
ax.scatter(nn_reference_acceptance, nn_reference_rejection, s=50, marker="D",
           color="#7b3294", edgecolor="white", zorder=4)
for cut, x, y_point in zip(xgb_reference_cuts, xgb_reference_acceptance, xgb_reference_rejection):
    ax.annotate(f">{cut:.2f}", (x, y_point), xytext=(5, 7),
                textcoords="offset points", fontsize=8, color="#2166ac")
for cut, x, y_point in zip(nn_reference_cuts, nn_reference_acceptance, nn_reference_rejection):
    ax.annotate(f">{cut:.2f}", (x, y_point), xytext=(5, -13),
                textcoords="offset points", fontsize=8, color="#7b3294")

ax.set(xlabel="Signal acceptance", ylabel="Prompt rejection",
       xlim=(0, 1.01), ylim=(0, 1.01),
       title="Detachment performance on held-out events\n(labels show classifier score cuts)")
ax.legend(loc="lower left")
ax.set_aspect("equal", adjustable="box")
plt.show()

### Pause before choosing a method or cut

How should this plot be read? The most useful region is toward the **upper-right**, where both signal acceptance and prompt rejection are high. There is no universally correct working point: moving left rejects more background but sacrifices more signal. An analysis should choose its operating point based on what limits the final measurement. Would you prefer a very pure but small sample, or a larger sample containing more background that must be modelled?

For this analysis, we choose to retain approximately **50% of the signal remaining after the loose detachment requirements**. The labelled points let you compare that choice with several looser and tighter score cuts. The score threshold is model-dependent: the next cell reports the XGBoost and neural-network cuts and the corresponding prompt rejection. At a fixed signal acceptance, the better method is the one whose point lies higher—rejecting more prompt background while keeping the same fraction of signal.

### What has machine learning gained?

<span style="color:green">**This plot shows us that we have gained a clear advantage by training these ML methods on the data**</span>. The XGBoost and neural-network curves both sit clearly above the points obtained by tightening only `tau_delta_z_sig`. **Either machine-learning method therefore gives substantially better signal–prompt separation than the single detachment-variable approach used in the Run 1 analysis.** The models have learned how to combine several imperfect pieces of geometrical information rather than relying on only one.

The two ML curves are also strikingly similar. Is that surprising? Discuss what it might imply about these six inputs and the complexity of the pattern being learned. One possibility is that both flexible models have extracted nearly all the useful information available in this small, physics-motivated feature set.

## Which inputs mattered?

XGBoost provides a built-in gain importance, measuring how much splits using each variable improve its objective. Neural networks do not have a directly equivalent built-in quantity because information is distributed across many weights and nonlinear combinations. Instead, we use **permutation importance**: shuffle one test feature at a time, breaking its relationship with the label, and measure the decrease in neural-network ROC AUC. These two importance scales answer related but not identical questions and *should not be compared numerically to each other*, but they can give us some useful sense of what inputs were most important to either model.

In [ ]:
working_points_50 = pd.DataFrame({
    "Model": ["XGBoost", "Neural network"],
    "Score cut": [xgb_cut_50, nn_cut_50],
    "Signal acceptance": [xgb_acceptance_50, nn_acceptance_50],
    "Prompt rejection": [xgb_rejection_50, nn_rejection_50],
}).set_index("Model")
display(working_points_50.style.format({
    "Score cut": "{:.4f}",
    "Signal acceptance": "{:.3f}",
    "Prompt rejection": "{:.3f}",
}))

# XGBoost gain importance.
xgb_importance = pd.Series(
    tuned_xgb.feature_importances_, index=FEATURES
).sort_values()

# Neural-network permutation importance, averaged over five shuffles.
baseline_nn_auc = roc_auc_score(y_test, nn_test_score)
permutation_rng = np.random.default_rng(RANDOM_STATE)
nn_importance_values = {}
for feature in FEATURES:
    auc_drops = []
    for _ in range(5):
        permuted = X_test.copy()
        permuted[feature] = permutation_rng.permutation(permuted[feature].to_numpy())
        permuted_scaled = tuned_nn_scaler.transform(permuted).astype("float32")
        permuted_score = tuned_nn.predict(permuted_scaled, batch_size=4096, verbose=0).ravel()
        auc_drops.append(baseline_nn_auc - roc_auc_score(y_test, permuted_score))
    nn_importance_values[feature] = np.mean(auc_drops)
nn_importance = pd.Series(nn_importance_values).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
xgb_importance.plot.barh(ax=axes[0], color="#2166ac")
nn_importance.plot.barh(ax=axes[1], color="#7b3294")
axes[0].set(xlabel="Gain importance", title="XGBoost")
axes[1].set(xlabel="Mean decrease in test ROC AUC", title="Neural network permutation importance")
fig.suptitle("Which physics-informed inputs matter most?", fontsize=16, y=1.02)
fig.tight_layout()
plt.show()

At the chosen operating point, the XGBoost requirement is approximately `score > 0.6664`, retaining 50.0% of signal and rejecting 85.6% of prompt. The neural-network requirement is approximately `score > 0.6517`, retaining 50.0% of signal and rejecting 85.4% of prompt. The performances are extremely close; XGBoost is marginally better at this particular working point, but the difference is not practically dramatic.

## Save the trained models

We save the XGBoost trees as JSON and the TensorFlow model in Keras format. The neural network also needs the means and scales learned from its training data, so those are saved separately. We additionally save the chosen 50%-signal score thresholds. Any later application must preserve the same feature names, order, and definitions.

In [ ]:
XGB_MODEL_FILE = Path("detachment_xgboost.json")
NN_MODEL_FILE = Path("detachment_neural_network.keras")
NN_SCALER_FILE = Path("detachment_neural_network_scaler.npz")
WORKING_POINT_FILE = Path("detachment_working_points.csv")

tuned_xgb.save_model(XGB_MODEL_FILE)
tuned_nn.save(NN_MODEL_FILE)
np.savez(
    NN_SCALER_FILE,
    mean=tuned_nn_scaler.mean_,
    scale=tuned_nn_scaler.scale_,
    features=np.array(FEATURES),
)
working_points_50.reset_index().to_csv(WORKING_POINT_FILE, index=False)

print(f"Saved XGBoost model: {XGB_MODEL_FILE.resolve()}")
print(f"Saved neural network: {NN_MODEL_FILE.resolve()}")
print(f"Saved NN preprocessing: {NN_SCALER_FILE.resolve()}")
print(f"Saved working points: {WORKING_POINT_FILE.resolve()}")

## Apply the detachment selections and inspect the $D_s^+\to3\pi$ region

Finally, we return to `data_faked.csv`, before any $\tau$-detachment requirements have been applied. We compare four stages:

1. no detachment selection;
2. only `tau_delta_z_sig > 2` and `tau_delta_r_sig > 0`;
3. the loose requirements followed by the XGBoost 50%-signal working point;
4. the loose requirements followed by the neural-network 50%-signal working point.

Like in the intro activity, we can fit each mass spectrum with a Gaussian $D_s^+$ peak plus a falling exponential background,

$$
f(m)=A\exp\left[-\frac{(m-\mu)^2}{2\sigma^2}\right]
+B\exp[-\lambda(m-1800)],
$$

allowing us to estimate the number of reconstructed $D_s^+\to3\pi$ decays at each stage. This is a deliberately simple teaching fit rather than the full analysis model.

In [ ]:
DATA_FILE = Path("data_faked.csv")
MASS_COLUMN = "m_3pi"
MASS_RANGE = (1800.0, 2100.0)  # MeV/c^2
N_MASS_BINS = 100
RAW_CUT_COLUMNS = ["tau_delta_z_sig", "tau_delta_r_sig"]

data_sample = pd.read_csv(
    DATA_FILE, usecols=[MASS_COLUMN, *RAW_CUT_COLUMNS, *FEATURES]
)
mass_window = data_sample[MASS_COLUMN].between(*MASS_RANGE, inclusive="left")
loose_mask = (
    (data_sample["tau_delta_z_sig"] > 2.0)
    & (data_sample["tau_delta_r_sig"] > 0.0)
)

# Evaluate the two classifiers only after the loose cuts, where the log inputs exist.
loose_data = data_sample.loc[loose_mask].copy()
loose_data["xgb_score"] = tuned_xgb.predict_proba(loose_data[FEATURES])[:, 1]
loose_scaled = tuned_nn_scaler.transform(loose_data[FEATURES]).astype("float32")
loose_data["nn_score"] = tuned_nn.predict(loose_scaled, batch_size=4096, verbose=0).ravel()

mass_samples = {
    "No detachment cuts": data_sample.loc[mass_window, MASS_COLUMN].to_numpy(),
    "Loose detachment cuts": loose_data.loc[
        loose_data[MASS_COLUMN].between(*MASS_RANGE, inclusive="left"), MASS_COLUMN
    ].to_numpy(),
    "Loose + XGBoost (50% signal)": loose_data.loc[
        loose_data[MASS_COLUMN].between(*MASS_RANGE, inclusive="left")
        & (loose_data["xgb_score"] > xgb_cut_50), MASS_COLUMN
    ].to_numpy(),
    "Loose + neural net (50% signal)": loose_data.loc[
        loose_data[MASS_COLUMN].between(*MASS_RANGE, inclusive="left")
        & (loose_data["nn_score"] > nn_cut_50), MASS_COLUMN
    ].to_numpy(),
}

mass_bins = np.linspace(MASS_RANGE[0], MASS_RANGE[1], N_MASS_BINS + 1)
mass_centers = 0.5 * (mass_bins[:-1] + mass_bins[1:])
bin_width = mass_bins[1] - mass_bins[0]

def gaussian_plus_exponential(mass, amplitude, mean, sigma, background, slope):
    gaussian = amplitude * np.exp(-0.5 * ((mass - mean) / sigma) ** 2)
    exponential = background * np.exp(-slope * (mass - MASS_RANGE[0]))
    return gaussian + exponential

fit_results = []
fitted_curves = {}
for label, masses in mass_samples.items():
    counts, _ = np.histogram(masses, bins=mass_bins)
    uncertainties = np.sqrt(np.maximum(counts, 1.0))
    background_guess = max(float(np.median(counts[:15])), 1.0)
    amplitude_guess = max(float(counts.max() - background_guess), 1.0)
    parameters, covariance = curve_fit(
        gaussian_plus_exponential,
        mass_centers,
        counts,
        p0=[amplitude_guess, 1968.0, 10.0, background_guess, 0.001],
        sigma=uncertainties,
        absolute_sigma=True,
        bounds=([0, 1945, 2, 0, 0], [np.inf, 1990, 35, np.inf, 0.02]),
        maxfev=30000,
    )
    amplitude, mean, sigma, background, slope = parameters
    yield_factor = np.sqrt(2 * np.pi) / bin_width
    ds_yield = yield_factor * amplitude * sigma
    yield_variance = yield_factor**2 * (
        sigma**2 * covariance[0, 0]
        + amplitude**2 * covariance[2, 2]
        + 2 * amplitude * sigma * covariance[0, 2]
    )
    ds_yield_error = np.sqrt(max(yield_variance, 0.0))
    fitted_curves[label] = parameters
    fit_results.append({
        "Selection": label,
        "Events in window": len(masses),
        "Fitted Ds yield": ds_yield,
        "Yield uncertainty": ds_yield_error,
        "Peak mean [MeV/c²]": mean,
        "Peak width [MeV/c²]": sigma,
    })

selection_colors = ["#666666", "#e6a700", "#2166ac", "#7b3294"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
for ax, (label, masses), color in zip(axes.flat, mass_samples.items(), selection_colors):
    density, _, _ = ax.hist(
        masses, bins=mass_bins, density=False, histtype="step", linewidth=2,
        color=color, label=f"Data ({len(masses):,} events)",
    )
    parameters = fitted_curves[label]
    total_fit = gaussian_plus_exponential(mass_centers, *parameters)
    gaussian_fit = parameters[0] * np.exp(
        -0.5 * ((mass_centers - parameters[1]) / parameters[2]) ** 2
    )
    ax.plot(mass_centers, total_fit, color="black", lw=1.8,
            label="Gaussian + exponential fit")
    ax.plot(mass_centers, gaussian_fit, color="#2a9d8f", lw=1.5,
            linestyle="--", label=r"Fitted $D_s^+$ component")
    ax.set(title=label, ylabel="Events per bin")
    ax.legend(fontsize=9)
for ax in axes[-1, :]:
    ax.set_xlabel(r"$m(3\pi)$ [MeV/$c^2$]")
fig.suptitle(r"The $D_s^+\to3\pi$ region through the detachment selection", fontsize=16, y=1.01)
fig.tight_layout()
plt.show()

fit_results = pd.DataFrame(fit_results).set_index("Selection")
display(fit_results.style.format({
    "Events in window": "{:,}",
    "Fitted Ds yield": "{:.0f}",
    "Yield uncertainty": "{:.0f}",
    "Peak mean [MeV/c²]": "{:.2f}",
    "Peak width [MeV/c²]": "{:.2f}",
}))

## Summary

The loose geometric requirements already reveal the $D_s^+$ peak more clearly by suppressing prompt production. The multivariate detachment classifiers then produce a much purer displaced sample: although the total number of reconstructed $D_s^+$ decays decreases with tighter selection, the peak becomes far more prominent relative to the smooth background. This then makes it possible for a selected $D_s^+$ control region to be useful for understanding the double-charm contribution later in the analysis.

In this activity, XGBoost and the neural network perform almost identically. XGBoost gives slightly better prompt rejection at the chosen 50%-signal working point, while both substantially outperform a single tighter cut on longitudinal displacement.

More broadly, we learned about the balance between leveraging powerful ML methods with traditional physics insights, and then how best to combine the two and effectively train ML classifiers. Throwing in every low-level quantity can work, but carefully designed variables can express the relevant geometry much more directly, while genuinely random inputs add no physical information and may slightly hinder learning. We also saw that hyperparameters control a model's flexibility and training, ROC curves expose the trade-off between keeping signal and rejecting background, and the “best” score cut depends on the needs of the analysis. Finally, applying the chosen classifiers to the data connected those statistical ideas back to a visible physics result: a cleaner $D_s^+\to3\pi$ peak that can be used to study an important background.

## Credits
This notebook was created by Alex Fernez, largely based on the real detachment BDT developed by [the published Run 2 $R(D^*)$ hadronic-$\tau$ analysis](https://www.google.com/url?q=https%3A%2F%2Farxiv.org%2Fpdf%2F2305.01463), and using simulation from LHCb (also borrowed from that analysis).